In [ ]:
# ===============================
# SARIMA vs SARIMAX Demonstration
# ===============================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --------------------------------
# 1. Load dataset
# --------------------------------

# AirPassengers dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
df = pd.read_csv(url, parse_dates=['Month'], index_col='Month')

# Rename column for convenience
df.columns = ['Passengers']

# --------------------------------
# 2. Create Exogenous Variable
# --------------------------------
# Fake example: advertising spend (random but trending)

np.random.seed(42)
df['Ad_Spend'] = np.linspace(10, 50, len(df)) + np.random.normal(0, 2, len(df))

# --------------------------------
# 3. Train-Test Split
# --------------------------------

train = df.iloc[:120]  # first 10 years
test = df.iloc[120:]   # remaining data

# --------------------------------
# 4. SARIMA Model
# SARIMA(p,d,q)(P,D,Q)m
# --------------------------------
# For monthly data → m = 12
# Choosing example order: (1,1,1)(1,1,1,12)

sarima_model = SARIMAX(
    train['Passengers'],
    order=(1,1,1),                 # (p,d,q)
    seasonal_order=(1,1,1,12),     # (P,D,Q,m)
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_result = sarima_model.fit()

# Forecast
sarima_forecast = sarima_result.forecast(steps=len(test))

# --------------------------------
# 5. SARIMAX Model (with exogenous variable)
# --------------------------------

sarimax_model = SARIMAX(
    train['Passengers'],
    exog=train[['Ad_Spend']],      # <-- X variable
    order=(1,1,1),
    seasonal_order=(1,1,1,12),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarimax_result = sarimax_model.fit()

# Forecast (IMPORTANT: must provide future exog values)
sarimax_forecast = sarimax_result.forecast(
    steps=len(test),
    exog=test[['Ad_Spend']]
)

# --------------------------------
# 6. Evaluation Metrics
# --------------------------------

def evaluate(true, pred, model_name):
    mae = mean_absolute_error(true, pred)
    rmse = np.sqrt(mean_squared_error(true, pred))
    print(f"\n{model_name} Performance")
    print("MAE :", round(mae, 2))
    print("RMSE:", round(rmse, 2))

evaluate(test['Passengers'], sarima_forecast, "SARIMA")
evaluate(test['Passengers'], sarimax_forecast, "SARIMAX")

# --------------------------------
# 7. Plot Results
# --------------------------------

plt.figure(figsize=(12,6))
plt.plot(train['Passengers'], label="Train")
plt.plot(test['Passengers'], label="Test")
plt.plot(test.index, sarima_forecast, label="SARIMA Forecast")
plt.plot(test.index, sarimax_forecast, label="SARIMAX Forecast")
plt.legend()
plt.title("SARIMA vs SARIMAX Forecast")
plt.show()